Generating arrays for h5 file

In [1]:
import glob
import h5py
import importlib
import IPython.display as ipd
import soxr
import numpy as np
import os
import pandas as pd
import pickle
import soundfile as sf
import src.audio_transforms as at
import src.custom_modules as cm
import src.spatial_attn_lightning as binaural_lightning
importlib.reload(binaural_lightning)
import sys
import torch
import tqdm
import yaml

from pathlib import Path
from pytorch_lightning import Trainer
from scipy import signal
from scipy.io.wavfile import read, write

sys.path.append('../')
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

In [4]:
# path to 376 wiki words
swc_path = '/om2/user/msaddler/spatial_audio_pipeline/assets/human_experiment_v00/foreground_swc/'

# path o common voice words not included in training
df = pd.read_pickle('/om2/user/imgriff/datasets/spatial_audio_pipeline/assets/commonvoice_9_en/manifest_all_word_alignments.pdpkl')

In [5]:
manifest = pd.read_pickle(swc_path + 'manifest.pdpkl')

In [6]:
manifest

,client_id,clip_dur_in_s,clip_end_in_s,clip_start_in_s,corpus,raw_clip_dur_in_s,raw_clip_end_in_s,raw_clip_start_in_s,raw_src_fn,raw_total_file_duration_in_s,split,split_int,sr,src_fn,total_file_duration_in_s,word
0,a-c-norman,3.0,3.0,0.0,swc,0.32,2094.94,2094.62,/scratch2/weka/mcdermott/msaddler/swc/english/...,2175.444172,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,above
1,jebjoya,3.0,3.0,0.0,swc,0.49,1715.87,1715.38,/scratch2/weka/mcdermott/msaddler/swc/english/...,2793.356190,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,according
2,caninedoubletake,3.0,3.0,0.0,swc,0.36,169.03,168.67,/scratch2/weka/mcdermott/msaddler/swc/english/...,987.438730,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,across
3,karltalk,3.0,3.0,0.0,swc,0.60,2429.51,2428.91,/scratch2/weka/mcdermott/msaddler/swc/english/...,4802.892336,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,action
4,s-whistler,3.0,3.0,0.0,swc,0.80,1149.87,1149.07,/scratch2/weka/mcdermott/msaddler/swc/english/...,4463.715556,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,activities
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
371,incledon,3.0,3.0,0.0,swc,0.69,231.72,231.03,/scratch2/weka/mcdermott/msaddler/swc/english/...,273.573152,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,world
372,ama1016,3.0,3.0,0.0,swc,0.42,193.47,193.05,/scratch2/weka/mcdermott/msaddler/swc/english/...,273.502041,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,writing
373,zanimum,3.0,3.0,0.0,swc,0.27,13.92,13.65,/scratch2/weka/mcdermott/msaddler/swc/english/...,452.154921,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,written
374,tonyle,3.0,3.0,0.0,swc,0.31,1208.98,1208.67,/scratch2/weka/mcdermott/msaddler/swc/english/...,2359.540680,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,wrote


In [7]:
fn_pkl_src = '/scratch2/weka/mcdermott/raygon/projects/public/jsinDataset/assets/data/interim/swc/mungedFinalDataframeWords_swc_readerNormalized_sexNormalized_accent.pdpkl'
fn_pkl_dst = '/om2/user/msaddler/spatial_audio_pipeline/assets/swc/manifest_all_words.pdpkl'

In [10]:
word_dict = pickle.load(open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_800_word_label_to_int_dict.pkl", 'rb'))

In [12]:
words = list(word_dict.keys())

In [52]:
words = [word.replace("'", "") for word in words]

In [61]:
manifest_all_words = pd.read_pickle(fn_pkl_dst)

In [62]:
# filter out words not in 'words' list
manifest_all_words = manifest_all_words[manifest_all_words['word'].isin(words)]

In [63]:
len(manifest_all_words['word'].unique())

791

In [64]:
word_counts = manifest_all_words['word'].value_counts()

In [82]:
# try to get one male and one female example for each word:

word_ixs = []
for word in manifest_all_words['word'].unique():
    word_1 = manifest_all_words[manifest_all_words['word'] == word].index[0]
    word_ixs.append(word_1)
    gender = manifest_all_words.iloc[word_1]['gender']
    try:
        word_opposite_gender = manifest_all_words[(manifest_all_words['word'] == word) & (manifest_all_words['gender'] != gender)].index[0]
        word_ixs.append(word_opposite_gender)
    except:
        word_ixs.append(manifest_all_words[(manifest_all_words['word'] == word) & (manifest_all_words['client_id'] != manifest_all_words.iloc[word_1]['client_id'])].index[0])

In [89]:
down_df = manifest_all_words.groupby(['word', 'gender']).sample(1, replace=False)

In [91]:
final_df = down_df.groupby('word').filter(lambda x: len(x) == 2)

In [98]:
final_df = final_df.reset_index().rename(columns={'index':'src_ix'})

In [100]:
final_df.drop(columns=['level_0',], inplace=True)

In [113]:
final_df

,src_ix,client_id,clip_dur_in_s,clip_end_in_s,clip_start_in_s,corpus,gender,gender_int,split,split_int,sr,src_fn,total_file_duration_in_s,word
0,762604,theexistentialist383,0.29,945.87,945.58,swc,female,0,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,1049.035465,about
1,678071,oblong1,0.12,395.14,395.02,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,507.740590,about
2,135563,popularoutcast,0.51,58.15,57.64,swc,female,0,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,1160.012336,above
3,515363,macropode,0.26,330.70,330.44,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,1292.804354,above
4,712025,flyingtoaster,0.46,3282.14,3281.68,swc,female,0,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,3524.873288,access
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1573,613229,anaxial,0.67,621.31,620.64,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,2108.315283,yellow
1574,139808,popularoutcast,0.38,1075.86,1075.48,swc,female,0,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,1434.382222,young
1575,265162,alexkillby,0.31,404.81,404.50,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,749.550295,young
1576,497651,persian-poet-gal,0.36,153.41,153.05,swc,female,0,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,3092.990839,younger


In [112]:
final_df.to_pickle('binaural_test_foreground_manifest.pdpkl')

In [108]:
all_words_not_filtered = pd.read_pickle(fn_pkl_dst)
oov_words = all_words_not_filtered[~all_words_not_filtered['word'].isin(words)]

In [109]:
talkers = final_df['client_id'].unique()
oov_words[oov_words['client_id'].isin(talkers)].client_id.value_counts()

s-whistler               60197
matthewdgonzalez         49291
mangst                   36135
the-voice-of-hassocks    35187
alexkillby               26728
                         ...  
donnyj                      49
puffin                      46
ralmin                      31
antasevaalya                25
linuxrox                     8
Name: client_id, Length: 224, dtype: int64

In [110]:
samples_per_talker = {talker:count for talker,count in final_df.client_id.value_counts().items()}
viables_cues = oov_words[oov_words.client_id.isin(talkers)]

cues = viables_cues.groupby('client_id').apply(lambda group: group.sample(samples_per_talker[group.name]))
cues.drop(columns='client_id', inplace=True)
cues = cues.reset_index()
cues.rename(columns={'level_1':'cue_src_ix'}, inplace=True)

In [115]:
final_df.sort_values(by='client_id', inplace=True)
final_df.reset_index(inplace=True, drop=True)


cues.sort_values(by='client_id', inplace=True)
cues.reset_index(inplace=True, drop=True)



### Merge cues with foregrounds  
cues[['cue_word', 'cue_src_ix', 'cue_client_id']] = cues[['word', 'cue_src_ix', 'client_id']]
# Combine as experiment dataframe
training_df = final_df.join(cues[['cue_word', 'cue_src_ix', 'cue_client_id']])
assert (training_df['client_id'] == training_df['cue_client_id']).all(), "Cue and Target talkers don't match!"

In [117]:
training_df.to_pickle('binaural_test_manifest.pdpkl')